# ScanOps V12 — 3B QLoRA 학습 (OWASP 0%, 과적합 차단)

**V11과의 결정적 차이:** 학습 데이터에 OWASP Benchmark 코드를 **한 건도** 넣지 않는다.
OWASP·CVEfixes는 순수 zero-shot 평가셋으로만 쓴다 → 데이터 누수 없음.

학습셋: `lora_train_v12_clean.jsonl` (264개, 16개 언어, 안전 44%, OWASP 0).
검증셋: `lora_train_v12_clean_val.jsonl` (분리된 val → 매 epoch val loss로 과적합 감시).

**런타임 유형을 반드시 `T4 GPU`로 바꾸세요** (수정 > 노트 설정 > 하드웨어 가속기 > T4).


## ① 의존성 설치 + GPU 확인


In [ ]:
!pip -q install transformers peft datasets accelerate bitsandbytes matplotlib
import torch
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0))

## ② 파일 업로드
`lora_train_v12_clean.jsonl` 과 `lora_train_v12_clean_val.jsonl` **두 개**를 선택해 올린다.


In [ ]:
import os
for f in ['lora_train_v12_clean.jsonl', 'lora_train_v12_clean_val.jsonl']:
    if os.path.exists(f): os.remove(f)
from google.colab import files
files.upload()   # 두 파일 모두 선택
print('학습:', os.path.exists('lora_train_v12_clean.jsonl'),
      '| 검증:', os.path.exists('lora_train_v12_clean_val.jsonl'))

## ③ 3B QLoRA 학습 (~30분, 3 epoch)
매 epoch **val loss를 측정**한다. train loss는 내려가는데 val loss가 올라가면 과적합 신호 → epoch를 줄인다.


In [ ]:
import torch, gc
try: del trainer, model
except: pass
gc.collect(); torch.cuda.empty_cache()

import json, time, os
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments)

TRAIN = 'lora_train_v12_clean.jsonl'
VAL   = 'lora_train_v12_clean_val.jsonl'
BASE  = 'Qwen/Qwen2.5-Coder-3B-Instruct'
# ★ V11과 동일 하이퍼파라미터, 단 작은 clean 데이터셋이라 EPOCHS=3
MAXLEN, R, ALPHA, EPOCHS, LR = 768, 32, 64, 3, 1e-4
SYS = 'You are a security code analyzer.'

def load(p): return [json.loads(l) for l in open(p) if l.strip()]
tr_rows, va_rows = load(TRAIN), load(VAL)
def safe_pct(rows): return 100*sum(1 for r in rows if r['completion'].upper().startswith('VULNERABILITY: NONE'))/len(rows)
print(f'train {len(tr_rows)} (안전 {safe_pct(tr_rows):.1f}%) | val {len(va_rows)} (안전 {safe_pct(va_rows):.1f}%)')

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
def fmt(e): return {'text': f'<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n'+e['prompt']+'<|im_end|>\n<|im_start|>assistant\n'+e['completion']+'<|im_end|>'}
def tk(e):
    o=tok(e['text'],truncation=True,max_length=MAXLEN); o['labels']=o['input_ids'].copy(); return o
tr = Dataset.from_list(tr_rows).map(fmt).map(tk, remove_columns=['prompt','completion','text'])
va = Dataset.from_list(va_rows).map(fmt).map(tk, remove_columns=['prompt','completion','text'])

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type=TaskType.CAUSAL_LM))
model.print_trainable_parameters()

args = TrainingArguments(output_dir='out', num_train_epochs=EPOCHS, per_device_train_batch_size=1,
    per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=LR,
    lr_scheduler_type='cosine', warmup_ratio=0.03, fp16=True, logging_steps=5,
    eval_strategy='epoch', save_strategy='no', report_to='none',
    gradient_checkpointing=True, optim='paged_adamw_8bit')
trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
    data_collator=DataCollatorForSeq2Seq(tok, pad_to_multiple_of=8, return_tensors='pt'))
t0=time.time(); trainer.train(); print(f'학습 {(time.time()-t0)/60:.1f}분')
model.save_pretrained('adapter'); tok.save_pretrained('adapter')
json.dump([e for e in trainer.state.log_history], open('adapter/train_log.json','w'), indent=2)

## ④ 학습곡선 + 과적합 점검
train loss와 val loss를 함께 그린다. 둘 다 수렴하면 OK. val loss가 어느 epoch부터 다시 **올라가면** 그 직전 epoch가 최적 → EPOCHS를 그 값으로 낮춰 재학습.


In [ ]:
import matplotlib.pyplot as plt, json
log = json.load(open('adapter/train_log.json'))
tr_l = [(e['step'], e['loss']) for e in log if 'loss' in e]
va_l = [(e['step'], e['eval_loss']) for e in log if 'eval_loss' in e]
plt.figure(figsize=(7,4))
if tr_l: plt.plot(*zip(*tr_l), label='train loss')
if va_l: plt.plot(*zip(*va_l), 'o-', label='val loss (epoch)')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.title('V12 learning curve (과적합 감시)')
plt.grid(alpha=.3); plt.savefig('adapter/v12_learning_curve.png', dpi=120, bbox_inches='tight'); plt.show()
if va_l:
    print('val loss per epoch:', [round(v,4) for _,v in va_l])
    if len(va_l)>=2 and va_l[-1][1] > min(v for _,v in va_l)+0.05:
        print('⚠️ val loss가 반등 → 과적합 신호. EPOCHS를 최저 val 지점으로 낮추세요.')
    else:
        print('✅ val loss 안정 → 과적합 신호 없음.')

## ⑤ 빠른 동작 점검 (취약/안전 각 1개)
학습한 어댑터가 취약/안전을 실제로 구별하는지 즉석 확인.


In [ ]:
import torch
tests = [
  ('Python', 'os.system("ping -c1 " + request.args.get("host"))'),         # 취약(CWE-78)
  ('Python', 'cur.execute("SELECT * FROM u WHERE id=%s", (uid,))'),          # 안전
]
model.eval()
for lang, code_snip in tests:
    prompt = f'Analyze this {lang} code for security vulnerabilities:\n\n```\n{code_snip}\n```\n\nRespond starting with VULNERABILITY:'
    msg = f'<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n'
    ids = tok(msg, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=40, do_sample=False, pad_token_id=tok.eos_token_id)
    print('CODE:', code_snip[:55])
    print('->', tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).split(chr(10))[0])
    print()

## ⑥ 어댑터 다운로드
`adapter.zip`을 받아 로컬에서 GGUF 변환 → Ollama 등록 (`scripts/convert_to_gguf_v12.py`).


In [ ]:
!cd adapter && zip -qr ../adapter_v12.zip . && cd ..
from google.colab import files
files.download('adapter_v12.zip')